## Markov Decision Processes (MDPs)A Markov Decision Process extends the basic Markov chain framework by introducing actions and rewards, enabling agents to make optimal decisions in stochastic environments. While standard Markov chains evolve passively according to fixed transition probabilities, MDPs allow an agent to choose actions that influence state transitions and generate rewards, forming the foundation for sequential decision-making under uncertainty.### MDP ComponentsFormally, an MDP is defined by the tuple $\left(\mathcal{S}, \mathcal{A}, R\left(s, a\right), T\left(s^{\prime}\,|\,s,a\right), \gamma\right)$:* **State space** $\mathcal{S}$: The set of all possible states $s$ that the system can occupy.* **Action space** $\mathcal{A}$: The set of all possible actions $a$ available to the agent. For a given state $s$, the available actions are denoted $\mathcal{A}_{s} \subseteq \mathcal{A}$.* **Reward function** $R\left(s, a\right)$: The immediate reward received when taking action $a$ in state $s$.* **Transition model** $T\left(s^{\prime}\,|\,s,a\right) = P(s_{t+1} = s^{\prime}\,|\,s_{t}=s,a_{t} = a)$: The probability that action $a$ in state $s$ at time $t$ results in state $s^{\prime}$ at time $t+1$.* **Discount factor** $\gamma \in [0,1]$: A parameter that weights future rewards relative to immediate rewards, with $\gamma = 0$ prioritizing immediate rewards and $\gamma \to 1$ valuing long-term returns.The agent's goal is to find a policy $\pi: \mathcal{S} \to \mathcal{A}$ that maps states to actions, $\pi(s) = a$, maximizing the expected cumulative discounted reward over time. Two common approaches for solving MDPs are Value Iteration and Random Rollout algorithms.___

## Value Iteration AlgorithmValue iteration computes the optimal value function $U^{*}(s)$ by iteratively applying the Bellman backup operation until convergence. The value function $U(s)$ represents the maximum expected cumulative discounted reward achievable from state $s$ under the optimal policy. At each iteration, the algorithm propagates value estimates backward from high-reward states, gradually refining the utility of each state until convergence.The Bellman update equation is:$$\begin{equation*}U_{k+1}(s) = \max_{a\in\mathcal{A}}\left(R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}}T(s^{\prime}\,|\,s,a)\cdot{U}_{k}(s^{\prime})\right)\end{equation*}$$As $k\to\infty$, the value function converges: $U_{k}(s) \to U^{*}(s)$. Once convergence is achieved, we extract the optimal policy $\pi^{*}(s)$ using the action-value function:$$\begin{equation*}Q^{*}(s,a) = R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}}T(s^{\prime}\,|\,s,a)\cdot{U^{*}}(s^{\prime})\end{equation*}$$The optimal policy selects the action that maximizes the action-value function:$$\begin{equation*}\pi^{*}(s) = \arg\max_{a\in\mathcal{A}}\,Q^{*}(s,a)\end{equation*}$$#### AlgorithmValue iteration algorithm for computing the optimal value function $U^{*}(s)$ and policy $\pi^{*}(s)$:__Initialize__: Given an MDP with state space $\mathcal{S}$, action space $\mathcal{A}$, reward function $R(s,a)$, transition model $T(s^{\prime}\,|\,s,a)$, discount factor $\gamma$, tolerance parameter $\epsilon$, and maximum number of iterations $T$. Initialize the iteration counter $k\gets 0$, the initial value function $U_{0}(s) \gets 0$ for all $s \in \mathcal{S}$, and $\texttt{converged}\gets\texttt{false}$.While $\texttt{converged}$ is $\texttt{false}$ __do__:1. For each state $s \in \mathcal{S}$, compute the updated value:   $$U_{k+1}(s) \gets \max_{a\in\mathcal{A}}\left(R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}}T(s^{\prime}\,|\,s,a)\cdot{U}_{k}(s^{\prime})\right)$$2. Check for convergence:    - If $\max_{s\in\mathcal{S}} \left|U_{k+1}(s) - U_{k}(s)\right| \leq \epsilon$, then set $\texttt{converged}\gets\texttt{true}$ and $U^{*}\gets{U}_{k+1}$.    - If $\max_{s\in\mathcal{S}} \left|U_{k+1}(s) - U_{k}(s)\right| > \epsilon$, update $k\gets{k+1}$ and $U_{k}\gets{U}_{k+1}$.3. Update the $\texttt{converged}$ flag:    - If $k\geq{T}$, then set $\texttt{converged}\gets\texttt{true}$ and $U^{*}\gets{U}_{k+1}$.__Extract Policy__: For each state $s \in \mathcal{S}$, compute:$$\pi^{*}(s) \gets \arg\max_{a\in\mathcal{A}}\left(R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}}T(s^{\prime}\,|\,s,a)\cdot{U^{*}}(s^{\prime})\right)$$This algorithm guarantees convergence to the optimal policy for finite MDPs and provides a principled approach to sequential decision-making in uncertain environments.___

## Random Rollout AlgorithmThe random rollout algorithm estimates the value of states by simulating random trajectories from a starting state and computing the cumulative discounted reward obtained along each path. Unlike value iteration, which requires knowledge of the transition model $T(s^{\prime}\,|\,s,a)$ and computes values for all states simultaneously, random rollout is a model-free sampling approach that explores the state space through direct interaction with the environment.The algorithm generates trajectories by selecting random actions at each state, transitioning according to the environment dynamics, and accumulating discounted rewards. By averaging the returns from multiple rollouts starting from a given state $s$, we obtain an empirical estimate of the value function $\hat{U}(s)$.#### AlgorithmRandom rollout algorithm for estimating the value function $\hat{U}(s)$ through Monte Carlo sampling:__Initialize__: Given an MDP with state space $\mathcal{S}$, action space $\mathcal{A}$, reward function $R(s,a)$, discount factor $\gamma$, maximum depth $d$, and number of rollouts $N$. Initialize the value estimates $\hat{U}(s) \gets 0$ and visit counts $n(s) \gets 0$ for all $s \in \mathcal{S}$.For $i = 1$ to $N$ __do__:1. Initialize the rollout:    - Set starting state $s_{0}$ (chosen uniformly or from a start distribution).    - Set depth counter $t\gets 0$, cumulative return $G\gets 0$, and visited states $\mathcal{V}\gets\{s_{0}\}$.    - Set current state $s \gets s_{0}$ and $\texttt{terminated}\gets\texttt{false}$.2. While $\texttt{terminated}$ is $\texttt{false}$ __do__:    - Select action $a$ uniformly at random from $\mathcal{A}_{s}$.    - Observe reward $r = R(s, a)$ and update cumulative return: $G \gets G + \gamma^{t} \cdot r$.    - Execute action $a$ and observe next state $s^{\prime}$.    - Update depth: $t \gets t + 1$.    - Check termination conditions:        - If $t \geq d$ (maximum depth reached), set $\texttt{terminated}\gets\texttt{true}$.        - If $s^{\prime} \in \mathcal{V}$ (cycle detected), set $\texttt{terminated}\gets\texttt{true}$.        - If $s^{\prime}$ is an absorbing state (terminal state), set $\texttt{terminated}\gets\texttt{true}$.        - Otherwise, update $\mathcal{V} \gets \mathcal{V} \cup \{s^{\prime}\}$ and $s\gets s^{\prime}$.3. Update value estimates:    - Increment visit count: $n(s_{0}) \gets n(s_{0}) + 1$.    - Update value estimate using incremental mean:       $$\hat{U}(s_{0}) \gets \hat{U}(s_{0}) + \frac{1}{n(s_{0})}\left(G - \hat{U}(s_{0})\right)$$__Output__: The estimated value function $\hat{U}(s)$ for all states visited during the $N$ rollouts.#### Convergence PropertiesRandom rollout converges to the true value function $U(s)$ as $N \to \infty$ without requiring knowledge of the transition model. By averaging empirical returns from multiple rollouts, the sample mean converges to the expected cumulative discounted reward. This model-free property forms the foundation of Monte Carlo methods in reinforcement learning, enabling agents to learn from direct experience in stochastic environments.___